# Tarea 3 — PyTorch

**Nombre del integrante:** Ares Ramírez

**Cédula del integrante:** 30.382.924

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [1]:
import copy
import math
import os
import pickle

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

NETWORK_FILE = "pytorch_fashion_1"   # change the number to your notebook (1, 2 or 3)
DATA_DIR = os.path.join(os.getcwd(), "data")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_SIZE = 784
VALIDATION_SIZE = 5000               # samples taken from the training set for early stopping

# Set to True ONCE to train and save the four required networks (the menu opens afterwards).
# Set it back to False before the final "Restart & Run All".
AUTO_TRAIN_REQUIRED_NETWORKS = False

# Convention: menu / user-facing messages are in Spanish; training logs are in English.

print("PyTorch", torch.__version__, "| device:", DEVICE)

PyTorch 2.15.0+cpu | device: cpu


## 2. Datos (Fashion MNIST original y clases simplificadas)

In [2]:
# Fashion MNIST classes, in the native order of the dataset.
ORIGINAL_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                  "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Simplified classification: 0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag.
SIMPLIFIED_NAMES = ["Top", "Bottom", "Footwear", "Bag"]
SIMPLIFIED_MAP = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

_transform = transforms.ToTensor()   # pixels in [0, 1], no Normalize outside the model


class SimplifiedDataset(Dataset):
    """Wraps Fashion MNIST and remaps its 10 labels onto the 4 simplified classes."""

    def __init__(self, base, mapping):
        self.base = base
        self.mapping = mapping

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label = self.base[index]
        return image, int(self.mapping[label])


_dataset_cache = {}


def get_datasets(classification):
    """Returns (train_dataset, test_dataset, class_names) for 'original' or 'simplified'."""
    if classification not in _dataset_cache:
        train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=_transform)
        test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=_transform)
        if classification == "simplified":
            _dataset_cache[classification] = (SimplifiedDataset(train, SIMPLIFIED_MAP),
                                              SimplifiedDataset(test, SIMPLIFIED_MAP),
                                              SIMPLIFIED_NAMES)
        else:
            _dataset_cache[classification] = (train, test, ORIGINAL_NAMES)
    return _dataset_cache[classification]


def split_train_validation(dataset, seed=0):
    """Splits the training set into (training part, validation part)."""
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator).tolist()
    return Subset(dataset, indices[VALIDATION_SIZE:]), Subset(dataset, indices[:VALIDATION_SIZE])


def make_loaders(classification, batch_size, seed=0):
    train_set, test_set, class_names = get_datasets(classification)
    train_part, validation_part = split_train_validation(train_set, seed)
    loaders = {
        "train": DataLoader(train_part, batch_size=batch_size, shuffle=True, drop_last=True),
        "validation": DataLoader(validation_part, batch_size=512),
        "test": DataLoader(test_set, batch_size=512),
    }
    return loaders, class_names


## 3. Modelos (rectangular y convolucional)

In [3]:
def hidden_layer_sizes(architecture, hidden_layers, neurons_per_layer, output_size):
    """Neurons of each hidden layer (the linear-interpolation funnel for 'convolutional')."""
    if architecture == "rectangular":
        return [neurons_per_layer] * hidden_layers
    # math.floor(x + 0.5) rounds .5 upwards (Python's round() rounds ties to the even number).
    return [max(1, int(math.floor(INPUT_SIZE + (output_size - INPUT_SIZE) * i / (hidden_layers + 1) + 0.5)))
            for i in range(1, hidden_layers + 1)]


def build_model(hidden_sizes, output_size, dropout=0.0, batch_norm=True, input_normalization=True):
    """nn.Sequential starting with Flatten; only Linear, ReLU, BatchNorm1d and Dropout."""
    layers = [nn.Flatten()]
    if input_normalization:
        layers.append(nn.BatchNorm1d(INPUT_SIZE))     # input normalization inside the model
    previous = INPUT_SIZE
    for size in hidden_sizes:
        layers.append(nn.Linear(previous, size))
        if batch_norm:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        previous = size
    layers.append(nn.Linear(previous, output_size))
    return nn.Sequential(*layers)


def default_config(classification="original", architecture="rectangular"):
    return {
        "classification": classification,
        "architecture": architecture,
        "hidden_layers": 2,
        "neurons_per_layer": 512,
        "output_size": 10 if classification == "original" else 4,
        "optimizer": "adam",
        "learning_rate": 0.001,
        "batch_size": 128,
        "weight_decay": 0.0,
        "dropout": 0.0,
        "batch_norm": True,
        "input_normalization": True,
        "early_stopping": True,
        "patience": 5,
        "seed": 0,
    }


def structure_of(config):
    """Everything that forces a model rebuild when it changes."""
    return (config["architecture"], config["hidden_layers"], config["neurons_per_layer"],
            config["output_size"], config["dropout"], config["batch_norm"],
            config["input_normalization"])


class Network:
    """A model together with the configuration that produced it."""

    def __init__(self, model, config, epochs=0, test_accuracy=None, hidden_sizes=None):
        self.model = model
        self.config = config
        self.epochs = epochs
        self.test_accuracy = test_accuracy
        self.structure = structure_of(config)
        # Real sizes of the hidden layers (taken from the file when the network was loaded).
        if hidden_sizes is None:
            hidden_sizes = hidden_layer_sizes(config["architecture"], config["hidden_layers"],
                                              config["neurons_per_layer"], config["output_size"])
        self.hidden_sizes = list(hidden_sizes)
        # The optimizer lives with the network so momentum / Adam state survives between trainings.
        self.optimizer = None
        self.optimizer_key = None


## 4. Entrenar, probar, guardar y cargar

In [4]:
def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:                       # older PyTorch versions
        return torch.load(path, map_location="cpu")


def group_parameters(state_dict):
    """Groups the state dict keys by the index of their module inside the Sequential."""
    groups = {}
    for key in state_dict:
        groups.setdefault(int(key.split(".")[0]), []).append(key)
    return groups


def rebuild_model(state_dict, dropout=0.0):
    """Rebuilds the nn.Sequential from the shapes of a saved state dict.

    Dropout has no parameters, so it cannot be inferred from the shapes: pass its rate.
    """
    groups = group_parameters(state_dict)
    ordered = sorted(groups)
    tensors = {index: state_dict[groups[index][0]] for index in ordered}
    last_linear = max(index for index in ordered if tensors[index].dim() == 2)

    layers = [nn.Flatten()]
    key_map = {}
    seen_linear = False
    for position, index in enumerate(ordered):
        tensor = tensors[index]
        if tensor.dim() == 2:
            layers.append(nn.Linear(tensor.shape[1], tensor.shape[0]))
            seen_linear = True
        else:
            layers.append(nn.BatchNorm1d(tensor.shape[0]))
        key_map[index] = len(layers) - 1
        next_is_batch_norm = (position + 1 < len(ordered)
                              and tensors[ordered[position + 1]].dim() == 1)
        if tensor.dim() == 2:
            add_relu = index != last_linear and not next_is_batch_norm
        else:
            add_relu = seen_linear          # the input normalization is not followed by ReLU
        if add_relu:
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

    model = nn.Sequential(*layers)
    remapped = {}
    for index, keys in groups.items():
        for key in keys:
            remapped["%d.%s" % (key_map[index], key.split(".", 1)[1])] = state_dict[key]
    model.load_state_dict(remapped)
    return model


def check_contract(model):
    """The .pth contract: nn.Sequential that starts with Flatten and uses only 4 layer types."""
    allowed = (nn.Flatten, nn.Linear, nn.ReLU, nn.BatchNorm1d, nn.Dropout)
    if not isinstance(model, nn.Sequential) or not isinstance(model[0], nn.Flatten):
        raise ValueError("the model must be an nn.Sequential that starts with nn.Flatten()")
    for layer in model:
        if not isinstance(layer, allowed):
            raise ValueError("layer type not allowed in the .pth: %s" % type(layer).__name__)


def save_network(network, path):
    config = network.config
    check_contract(network.model)
    # Always save CPU tensors, so the file loads on machines without a GPU.
    cpu_state = {key: value.detach().cpu().clone()
                 for key, value in network.model.state_dict().items()}
    checkpoint = {
        "state_dict": cpu_state,
        "hyperparameters": {
            "architecture": config["architecture"],
            "hidden_layers": config["hidden_layers"],
            "neurons_per_layer": network.hidden_sizes,
            "epochs": network.epochs,
            "classification": config["classification"],
            "output_size": config["output_size"],
            "test_accuracy": network.test_accuracy,
            "optimizations": {
                "optimizer": config["optimizer"],
                "learning_rate": config["learning_rate"],
                "batch_size": config["batch_size"],
                "weight_decay": config["weight_decay"],
                "dropout": config["dropout"],
                "batch_norm": config["batch_norm"],
                "input_normalization": config["input_normalization"],
                "early_stopping": config["early_stopping"],
                "patience": config["patience"],
            },
        },
    }
    torch.save(checkpoint, path)


def load_network(path):
    checkpoint = _torch_load(path)
    if not isinstance(checkpoint, dict) or "state_dict" not in checkpoint:
        raise ValueError("el archivo no tiene el formato esperado (falta 'state_dict')")
    state_dict = checkpoint["state_dict"]
    hyperparameters = checkpoint.get("hyperparameters", {}) or {}
    optimizations = hyperparameters.get("optimizations", {}) or {}

    model = rebuild_model(state_dict, float(optimizations.get("dropout", 0.0) or 0.0))
    model.eval()

    groups = group_parameters(state_dict)
    ordered = sorted(groups)
    linear_sizes = [state_dict[groups[i][0]].shape[0] for i in ordered
                    if state_dict[groups[i][0]].dim() == 2]
    first_tensor = state_dict[groups[ordered[0]][0]]
    input_normalization = first_tensor.dim() == 1 and first_tensor.shape[0] == INPUT_SIZE
    batch_norm = sum(1 for i in ordered if state_dict[groups[i][0]].dim() == 1) > (
        1 if input_normalization else 0)

    config = default_config(hyperparameters.get("classification", "original"),
                            hyperparameters.get("architecture", "rectangular"))
    config["output_size"] = linear_sizes[-1]
    config["hidden_layers"] = len(linear_sizes) - 1
    config["neurons_per_layer"] = linear_sizes[0] if len(linear_sizes) > 1 else 0
    config["input_normalization"] = input_normalization
    config["batch_norm"] = batch_norm
    for key in ("optimizer", "learning_rate", "batch_size", "weight_decay", "dropout",
                "early_stopping", "patience"):
        if key in optimizations:
            config[key] = optimizations[key]

    network = Network(model, config, epochs=int(hyperparameters.get("epochs", 0)),
                      hidden_sizes=linear_sizes[:-1])
    accuracy = hyperparameters.get("test_accuracy")
    network.test_accuracy = None if accuracy is None else float(accuracy)
    return network


def make_optimizer(config, model):
    parameters = model.parameters()
    learning_rate = config["learning_rate"]
    weight_decay = config["weight_decay"]
    name = config["optimizer"]
    if name == "sgd":
        return torch.optim.SGD(parameters, lr=learning_rate, weight_decay=weight_decay)
    if name == "momentum":
        return torch.optim.SGD(parameters, lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=weight_decay)
    raise ValueError("unknown optimizer: %s" % name)


@torch.no_grad()
def evaluate_loss_accuracy(model, loader, criterion):
    """Mean loss and accuracy of a model over a loader."""
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        output = model(images)
        total_loss += criterion(output, labels).item() * images.size(0)
        correct += (output.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def predict(model, loader):
    """Returns (true labels, predicted labels) as numpy arrays."""
    model = model.to(DEVICE)               # a loaded network is on the CPU
    model.eval()
    predictions, targets = [], []
    for images, labels in loader:
        output = model(images.to(DEVICE))
        predictions.append(output.argmax(dim=1).cpu())
        targets.append(labels)
    return torch.cat(targets).numpy(), torch.cat(predictions).numpy()


def train_model(network, epochs, verbose=True):
    """Mini-batch cross-entropy training with optional early stopping. Returns the history."""
    config = network.config
    loaders, _ = make_loaders(config["classification"], config["batch_size"], config["seed"])
    model = network.model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    # Keep the same optimizer between trainings (momentum / Adam state); only recreate it
    # when the optimizer, the learning rate or the weight decay changed.
    optimizer_key = (config["optimizer"], config["learning_rate"], config["weight_decay"])
    if network.optimizer is None or network.optimizer_key != optimizer_key:
        network.optimizer = make_optimizer(config, model)
        network.optimizer_key = optimizer_key
    optimizer = network.optimizer

    history = {"train_loss": [], "validation_loss": [], "validation_accuracy": []}
    best_loss, best_state, waited = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0
        for images, labels in loaders["train"]:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            total += images.size(0)

        validation_loss, validation_accuracy = evaluate_loss_accuracy(
            model, loaders["validation"], criterion)
        history["train_loss"].append(total_loss / max(total, 1))
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)

        if verbose:
            print("  epoch %3d/%d | train loss %.4f | validation loss %.4f | validation accuracy %.2f%%"
                  % (epoch, epochs, history["train_loss"][-1], validation_loss,
                     100 * validation_accuracy))

        if config["early_stopping"]:
            if validation_loss < best_loss - 1e-4:
                best_loss = validation_loss
                best_state = copy.deepcopy(model.state_dict())
                waited = 0
            else:
                waited += 1
                if waited >= config["patience"]:
                    if verbose:
                        print("  early stopping after %d epochs without improvement" % waited)
                    break

    if config["early_stopping"] and best_state is not None:
        model.load_state_dict(best_state)
    network.model = model
    network.epochs += len(history["train_loss"])
    return history


def plot_history(history, title):
    """Line plot with the loss (and the validation accuracy) of every epoch."""
    epochs = range(1, len(history["train_loss"]) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.3))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="entrenamiento")
    axes[0].plot(epochs, history["validation_loss"], marker="s", label="validación")
    axes[0].set_title("Pérdida (loss) por época")
    axes[0].set_xlabel("época")
    axes[0].set_ylabel("entropía cruzada")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, [100 * value for value in history["validation_accuracy"]],
                 marker="o", color="tab:green")
    axes[1].set_title("Accuracy de validación por época")
    axes[1].set_xlabel("época")
    axes[1].set_ylabel("accuracy (%)")
    axes[1].grid(alpha=0.3)

    figure.suptitle(title)
    figure.tight_layout()
    return figure


def confusion_matrix(targets, predictions, n_classes):
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    return matrix


def metrics_from_confusion_matrix(matrix):
    """Accuracy, per-class precision/recall and their macro averages."""
    true_positive = np.diag(matrix).astype(float)
    support = matrix.sum(axis=1).astype(float)
    predicted = matrix.sum(axis=0).astype(float)
    precision = np.divide(true_positive, predicted, out=np.zeros_like(true_positive),
                          where=predicted > 0)
    recall = np.divide(true_positive, support, out=np.zeros_like(support), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(precision), where=(precision + recall) > 0)
    return {"accuracy": true_positive.sum() / max(matrix.sum(), 1),
            "correct": int(true_positive.sum()),
            "precision": precision, "recall": recall, "f1": f1, "support": support.astype(int),
            "macro_precision": precision.mean(), "macro_recall": recall.mean(),
            "macro_f1": f1.mean()}


def plot_confusion_matrix(matrix, class_names, accuracy):
    size = len(class_names)
    figure, axis = plt.subplots(figsize=(1.0 * size + 3.5, 0.9 * size + 2.5))
    image = axis.imshow(matrix, cmap="Blues")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    threshold = matrix.max() / 2.0 if matrix.max() > 0 else 0
    for row in range(size):
        for column in range(size):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center",
                      fontsize=9, color="white" if matrix[row, column] > threshold else "black")
    axis.set_xticks(range(size))
    axis.set_yticks(range(size))
    axis.set_xticklabels(class_names, rotation=45, ha="right")
    axis.set_yticklabels(class_names)
    axis.set_xlabel("clase predicha")
    axis.set_ylabel("clase real")
    axis.set_title("Matriz de confusión (accuracy %.2f%%)" % (100 * accuracy))
    figure.tight_layout()
    return figure


def print_metrics(metrics, class_names):
    print("\n  %-18s %10s %10s %10s %8s" % ("clase", "precisión", "recall", "f1", "soporte"))
    for name, precision, recall, f1, support in zip(class_names, metrics["precision"],
                                                    metrics["recall"], metrics["f1"],
                                                    metrics["support"]):
        print("  %-18s %9.2f%% %9.2f%% %9.2f%% %8d"
              % (name, 100 * precision, 100 * recall, 100 * f1, support))
    print("  %-18s %9.2f%% %9.2f%% %9.2f%% %8d"
          % ("macro", 100 * metrics["macro_precision"], 100 * metrics["macro_recall"],
             100 * metrics["macro_f1"], metrics["support"].sum()))
    print("  accuracy: %.2f%%   (%d aciertos de %d)"
          % (100 * metrics["accuracy"], metrics["correct"], metrics["support"].sum()))


REQUIRED_RECIPES = [
    ("original", "rectangular", 15, {"hidden_layers": 2, "neurons_per_layer": 512,
                                     "optimizer": "adamw", "learning_rate": 0.001,
                                     "weight_decay": 1e-4, "batch_size": 128,
                                     "dropout": 0.2, "batch_norm": True,
                                     "input_normalization": True}),
    ("original", "convolutional", 15, {"hidden_layers": 2, "optimizer": "adamw",
                                       "learning_rate": 0.001, "weight_decay": 1e-4,
                                       "batch_size": 128, "dropout": 0.1,
                                       "batch_norm": True, "input_normalization": True}),
    ("simplified", "rectangular", 10, {"hidden_layers": 2, "neurons_per_layer": 256,
                                       "optimizer": "adam", "learning_rate": 0.001,
                                       "batch_size": 128, "dropout": 0.1,
                                       "batch_norm": True, "input_normalization": True}),
    ("simplified", "convolutional", 10, {"hidden_layers": 2, "optimizer": "adam",
                                         "learning_rate": 0.001, "batch_size": 128,
                                         "dropout": 0.1, "batch_norm": True,
                                         "input_normalization": True}),
]


def train_required_networks():
    for classification, architecture, epochs, settings in REQUIRED_RECIPES:
        print("\n===== %s / %s =====" % (classification, architecture))
        config = default_config(classification, architecture)
        config.update(settings)
        sizes = hidden_layer_sizes(config["architecture"], config["hidden_layers"],
                                   config["neurons_per_layer"], config["output_size"])
        torch.manual_seed(config["seed"])
        network = Network(build_model(sizes, config["output_size"], config["dropout"],
                                      config["batch_norm"], config["input_normalization"]),
                          config)
        print("  layers: %s" % " -> ".join(str(n) for n in [INPUT_SIZE] + sizes + [config["output_size"]]))
        train_model(network, epochs)

        loaders, class_names = make_loaders(classification, config["batch_size"], config["seed"])
        targets, predictions = predict(network.model, loaders["test"])
        matrix = confusion_matrix(targets, predictions, len(class_names))
        metrics = metrics_from_confusion_matrix(matrix)
        network.test_accuracy = float(metrics["accuracy"])

        path = "%s_%s_%s.pth" % (NETWORK_FILE, classification, architecture)
        save_network(network, path)
        print("  test accuracy %.2f%%  ->  %s" % (100 * metrics["accuracy"], path))

        # Check that the saved file loads again and gives the same predictions.
        reloaded = load_network(path)
        _, reloaded_predictions = predict(reloaded.model, loaders["test"])
        reloaded_accuracy = float((reloaded_predictions == targets).mean())
        print("  reload check: %.2f%% (must match %.2f%%)"
              % (100 * reloaded_accuracy, 100 * metrics["accuracy"]))
        threshold = 0.75 if classification == "original" else 0.80
        if metrics["accuracy"] <= threshold:
            print("  WARNING: accuracy is not above the %.0f%% needed for full marks; train longer."
                  % (100 * threshold))


## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
class App:
    """Interactive console menu: create, load, train, test and save networks."""

    def __init__(self):
        self.network = None

    # ---------------- user input helpers ----------------
    def ask_int(self, label, minimum, default=None):
        while True:
            suffix = "" if default is None else " [por defecto %d]" % default
            raw = input("  %s%s: " % (label, suffix)).strip()
            if raw == "" and default is not None:
                return default
            try:
                value = int(raw)
            except ValueError:
                print("    Escribe un número entero.")
                continue
            if value < minimum:
                print("    Debe ser mayor o igual a %d." % minimum)
                continue
            return value

    def ask_float(self, label, default=None, minimum=0.0, maximum=None):
        while True:
            suffix = "" if default is None else " [por defecto %g]" % default
            raw = input("  %s%s: " % (label, suffix)).strip().replace(",", ".")
            if raw == "" and default is not None:
                return default
            try:
                value = float(raw)
            except ValueError:
                print("    Escribe un número (por ejemplo 0.001).")
                continue
            if value < minimum or (maximum is not None and value > maximum):
                print("    Debe estar entre %g y %s." % (minimum, "infinito" if maximum is None else "%g" % maximum))
                continue
            return value

    def ask_choice(self, label, options, default):
        while True:
            raw = input("  %s [por defecto %s]: " % (label, default)).strip().lower()
            if raw == "":
                return default
            if raw in options:
                return raw
            print("    Opciones válidas: %s." % ", ".join(options))

    def ask_yes_no(self, label, default=True):
        while True:
            raw = input("  %s (s/n) [por defecto %s]: "
                        % (label, "s" if default else "n")).strip().lower()
            if raw == "":
                return default
            if raw in ("s", "si", "sí", "y", "yes"):
                return True
            if raw in ("n", "no"):
                return False
            print("    Responde s o n.")

    # ---------------- summary ----------------
    def print_summary(self):
        network = self.network
        config = network.config
        path = [str(INPUT_SIZE)] + [str(size) for size in network.hidden_sizes] + [str(config["output_size"])]
        print("\n---- Red actual ----")
        print("  clasificación     : %s (%d clases)" % (config["classification"], config["output_size"]))
        print("  arquitectura      : %s" % config["architecture"])
        print("  capas             : %s" % " -> ".join(path))
        print("  épocas acumuladas : %d" % network.epochs)
        print("  optimizaciones    : %s" % self.optimizations_text(config))
        if network.test_accuracy is not None:
            print("  último accuracy de prueba: %.2f%%" % (100 * network.test_accuracy))

    def optimizations_text(self, config):
        parts = [config["optimizer"], "lr=%g" % config["learning_rate"],
                 "lote=%d" % config["batch_size"]]
        if config["weight_decay"] > 0:
            parts.append("weight decay=%g" % config["weight_decay"])
        if config["dropout"] > 0:
            parts.append("dropout=%g" % config["dropout"])
        if config["batch_norm"]:
            parts.append("batch norm")
        if config["input_normalization"]:
            parts.append("normalización de entrada")
        if config["early_stopping"]:
            parts.append("early stopping (%d)" % config["patience"])
        return ", ".join(parts)

    # ---------------- creation / configuration ----------------
    def configure_optimizations(self, config, creating=False):
        previous = dict(config)          # to undo the changes if the user cancels a weight reset
        print("  (pulsa enter para mantener el valor por defecto)")
        config["optimizer"] = self.ask_choice("Optimizador (sgd / momentum / adam / adamw)",
                                              ["sgd", "momentum", "adam", "adamw"],
                                              config["optimizer"])
        config["learning_rate"] = self.ask_float("Tasa de aprendizaje", config["learning_rate"],
                                                 minimum=1e-6)
        config["batch_size"] = self.ask_int("Tamaño del lote (batch size)", 2, config["batch_size"])
        config["weight_decay"] = self.ask_float("Weight decay (regularización L2)",
                                                config["weight_decay"])
        config["batch_norm"] = self.ask_yes_no("Usar batch normalization", config["batch_norm"])
        config["input_normalization"] = self.ask_yes_no(
            "Normalizar la entrada (BatchNorm sobre los 784 píxeles)",
            config["input_normalization"])
        config["dropout"] = self.ask_float("Dropout (0 = desactivado)", config["dropout"],
                                           maximum=0.9)
        config["early_stopping"] = self.ask_yes_no("Usar early stopping (conjunto de validación)",
                                                   config["early_stopping"])
        if config["early_stopping"]:
            config["patience"] = self.ask_int("Paciencia del early stopping (épocas)", 1,
                                              config["patience"])
        config["seed"] = self.ask_int("Semilla aleatoria", 0, config["seed"])
        if not creating and self.network is not None:
            self.rebuild_if_needed(previous)

    def rebuild_if_needed(self, previous):
        network = self.network
        config = network.config
        new, old = structure_of(config), network.structure
        if new == old:
            return

        # Only the dropout rate changed (index 4): the weights can be kept.
        if new[:4] + new[5:] == old[:4] + old[5:]:
            state = {key: value.detach().cpu().clone()
                     for key, value in network.model.state_dict().items()}
            network.model = rebuild_model(state, config["dropout"])
            network.structure = new
            network.optimizer = None
            print("  Dropout actualizado; los pesos entrenados se conservan.")
            return

        # Batch norm / input normalization changed: the weights cannot be kept.
        if network.epochs > 0:
            question = ("Este cambio reinicia los pesos y se perderían %d épocas de "
                        "entrenamiento. ¿Continuar?" % network.epochs)
            if not self.ask_yes_no(question, False):
                config.clear()
                config.update(previous)
                print("  Cambio cancelado: se conserva la configuración anterior.")
                return
        torch.manual_seed(config["seed"])
        network.model = build_model(network.hidden_sizes, config["output_size"],
                                    config["dropout"], config["batch_norm"],
                                    config["input_normalization"])
        network.structure = new
        network.optimizer = None
        network.epochs = 0
        network.test_accuracy = None
        print("  Los pesos se reiniciaron porque cambió la estructura de la red.")

    def create_network(self):
        print("\n----- Crear una red nueva -----")
        classification = self.ask_choice(
            "Clasificación: original (10 clases) o simplified (4 clases)",
            ["original", "simplified"], "original")
        architecture = self.ask_choice(
            "Arquitectura: rectangular o convolutional (embudo por interpolación)",
            ["rectangular", "convolutional"], "rectangular")

        config = default_config(classification, architecture)
        config["hidden_layers"] = self.ask_int("Número de capas ocultas", 0, 2)
        if architecture == "rectangular":
            config["neurons_per_layer"] = self.ask_int("Neuronas por cada capa oculta", 1, 512)
        self.configure_optimizations(config, creating=True)

        sizes = hidden_layer_sizes(config["architecture"], config["hidden_layers"],
                                   config["neurons_per_layer"], config["output_size"])
        torch.manual_seed(config["seed"])
        model = build_model(sizes, config["output_size"], config["dropout"],
                            config["batch_norm"], config["input_normalization"])
        self.network = Network(model, config)
        path = [str(INPUT_SIZE)] + [str(size) for size in sizes] + [str(config["output_size"])]
        print("  Red creada: %s" % " -> ".join(path))

    def load_network_from_file(self):
        default = "%s_original_rectangular.pth" % NETWORK_FILE
        while True:
            raw = input("  Ruta del archivo .pth [enter = %s, 'x' = cancelar]: " % default)
            raw = raw.strip().strip('"').strip("'")
            if raw.lower() == "x":
                return False
            path = raw if raw else default
            try:
                self.network = load_network(path)
            except (OSError, ValueError, RuntimeError, KeyError, EOFError, IndexError,
                    pickle.UnpicklingError) as error:
                print("  No se pudo cargar la red:", error)
                continue
            print("  Red cargada desde", path)
            return True

    # ---------------- actions ----------------
    def train_action(self):
        epochs = self.ask_int("Número de épocas", 1, 10)
        print("  Entrenando en %s ..." % DEVICE)
        history = train_model(self.network, epochs)
        config = self.network.config
        plot_history(history, "Entrenamiento %s / %s"
                     % (config["classification"], config["architecture"]))
        print("  Épocas acumuladas: %d" % self.network.epochs)
        plt.show()

    def test_action(self):
        config = self.network.config
        loaders, class_names = make_loaders(config["classification"], config["batch_size"],
                                            config["seed"])
        print("  Evaluando %d imágenes de prueba ..." % len(loaders["test"].dataset))
        targets, predictions = predict(self.network.model, loaders["test"])
        matrix = confusion_matrix(targets, predictions, len(class_names))
        metrics = metrics_from_confusion_matrix(matrix)
        print_metrics(metrics, class_names)
        plot_confusion_matrix(matrix, class_names, metrics["accuracy"])
        self.network.test_accuracy = float(metrics["accuracy"])
        plt.show()

    def save_action(self):
        config = self.network.config
        suggested = "%s_%s_%s.pth" % (NETWORK_FILE, config["classification"],
                                      config["architecture"])
        raw = input("  Archivo donde guardar [enter = %s]: " % suggested).strip()
        path = raw.strip('"').strip("'") if raw else suggested
        if not path.lower().endswith(".pth"):
            path += ".pth"
        try:
            save_network(self.network, path)
        except OSError as error:
            print("  No se pudo guardar:", error)
            return
        print("  Red guardada en", path)

    # ---------------- menus ----------------
    def network_menu(self):
        while True:
            self.print_summary()
            print("\n1) Entrenar\n2) Probar contra el conjunto de prueba\n"
                  "3) Guardar la red\n4) Configurar optimizaciones\n5) Volver al menú principal")
            option = self.ask_choice("Elige (1-5)", ["1", "2", "3", "4", "5"], "5")
            if option == "1":
                self.train_action()
            elif option == "2":
                self.test_action()
            elif option == "3":
                self.save_action()
            elif option == "4":
                self.configure_optimizations(self.network.config)
            else:
                return

    def main_menu(self):
        while True:
            print("\n===== Redes neuronales con PyTorch: Fashion MNIST =====")
            print("1) Crear una red nueva\n2) Cargar una red desde un archivo .pth\n3) Salir")
            option = self.ask_choice("Elige (1/2/3)", ["1", "2", "3"], "3")
            if option == "1":
                self.create_network()
            elif option == "2":
                if not self.load_network_from_file():
                    continue
            else:
                print("Fin.")
                return
            self.network_menu()


def main():
    app = App()
    try:
        app.main_menu()
    except (EOFError, KeyboardInterrupt):
        print("\nNo hay entrada interactiva disponible; el menú se detuvo.")
    except Exception as error:
        # Running the notebook without an interactive kernel (for example nbconvert)
        # makes input() raise StdinNotImplementedError: that is not a bug of the program.
        if type(error).__name__ != "StdinNotImplementedError":
            raise
        print("\nNo hay entrada interactiva disponible; el menú se detuvo.")


if AUTO_TRAIN_REQUIRED_NETWORKS:
    train_required_networks()
main()


===== Redes neuronales con PyTorch: Fashion MNIST =====
1) Crear una red nueva
2) Cargar una red desde un archivo .pth
3) Salir
